In [1]:
import os
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import noisereduce as nr
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

import joblib

/home/habib/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df ={"discomfort":[], "hungry": [], "tired": []}
path = "/home/habib/mindcloud/project/dataset"
for folder in os.listdir(path):
    folder_path = os.path.join(path, folder)
    for file in os.listdir(folder_path):
        df[folder].append(os.path.join(folder_path, file))
cols = 2
rows = len(df["discomfort"]) + len(df["hungry"]) + len(df["tired"])
r=0
dataset = pd.DataFrame(None, index=range(rows), columns=["y", "x"], dtype=object)
for clss in df:
    for i in df[clss]:
        dataset.iloc[r, 0]=clss
        dataset.iloc[r, 1] = i
        r+=1
dataset = dataset.sample(frac=1)
print(dataset.head(50))
print(dataset.info())

              y                                                  x
67       hungry  /home/habib/mindcloud/project/dataset/hungry/8...
218      hungry  /home/habib/mindcloud/project/dataset/hungry/f...
357      hungry  /home/habib/mindcloud/project/dataset/hungry/F...
213      hungry  /home/habib/mindcloud/project/dataset/hungry/1...
6    discomfort  /home/habib/mindcloud/project/dataset/discomfo...
71       hungry  /home/habib/mindcloud/project/dataset/hungry/5...
408      hungry  /home/habib/mindcloud/project/dataset/hungry/7...
378      hungry  /home/habib/mindcloud/project/dataset/hungry/6...
358      hungry  /home/habib/mindcloud/project/dataset/hungry/8...
405      hungry  /home/habib/mindcloud/project/dataset/hungry/F...
379      hungry  /home/habib/mindcloud/project/dataset/hungry/1...
301      hungry  /home/habib/mindcloud/project/dataset/hungry/e...
49       hungry  /home/habib/mindcloud/project/dataset/hungry/9...
325      hungry  /home/habib/mindcloud/project/dataset/hungry/

In [4]:
datasetcopy = dataset.copy()
train_df, test_df = train_test_split(dataset, test_size=0.2, random_state=42)

In [7]:
print(test_df.info())
print(test_df.head(10))

<class 'pandas.DataFrame'>
Index: 87 entries, 10 to 410
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   y       87 non-null     object
 1   x       87 non-null     object
dtypes: object(2)
memory usage: 2.0+ KB
None
              y                                                  x
10   discomfort  /home/habib/mindcloud/project/dataset/discomfo...
209      hungry  /home/habib/mindcloud/project/dataset/hungry/d...
246      hungry  /home/habib/mindcloud/project/dataset/hungry/7...
404      hungry  /home/habib/mindcloud/project/dataset/hungry/7...
306      hungry  /home/habib/mindcloud/project/dataset/hungry/9...
161      hungry  /home/habib/mindcloud/project/dataset/hungry/a...
244      hungry  /home/habib/mindcloud/project/dataset/hungry/a...
362      hungry  /home/habib/mindcloud/project/dataset/hungry/6...
148      hungry  /home/habib/mindcloud/project/dataset/hungry/B...
315      hungry  /home/habib/mindcloud/project/datase

In [8]:
def apply_pitch_shift(y, sr):
    # Choose a random semitone shift between -2 and +2
    n_steps = np.random.uniform(-2.0, 2.0)
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)

def apply_time_stretch(y):
    # Choose a random speed factor between 0.9 and 1.1
    rate = np.random.uniform(0.9, 1.1)
    return librosa.effects.time_stretch(y, rate=rate)

def apply_add_noise(y):
    # Generate random noise matching the audio length
    noise = np.random.normal(0, y.std(), len(y))
    # Mix it in lightly (0.005 is a safe factor for light noise)
    noise_factor = 0.005 
    return y + noise_factor * noise

In [9]:
target_count = train_df['y'].value_counts()['hungry']
print(f"target count = {target_count}")
discomfort_count = train_df['y'].value_counts()['discomfort']
tired_count = train_df['y'].value_counts()['tired']
print(f"discomfort = {discomfort_count}")
print(f"tired = {tired_count}")

target count = 306
discomfort = 22
tired = 18


In [12]:
#create a folder to save the augmented (fake) audio files
output_dir = "/home/habib/mindcloud/project/dataset/augmented/"
os.makedirs(output_dir, exist_ok=True)
rare = ["discomfort", "tired"]
needed_discomfort = target_count - discomfort_count
needed_tired = target_count - tired_count
new_rows = []
for i, row in train_df.iterrows():
    y = row["y"]
    x = row["x"]
    if y in rare and y == "discomfort":
        audio_data, sr = librosa.load(x, sr=None)
        for j in range(13):
            y_aug = apply_pitch_shift(audio_data, sr)
            y_aug = apply_time_stretch(y_aug)
            y_aug = apply_add_noise(y_aug)
            new_filename = f"aug_{y}_{i}_{j}.wav"
            new_file_path = os.path.join(output_dir, new_filename)
            sf.write(new_file_path, y_aug, sr)
            new_rows.append({'y': y, 'x': new_file_path})
    if y in rare and y == "tired":
            audio_data, sr = librosa.load(x, sr=None)
            for j in range(16):
                y_aug = apply_pitch_shift(audio_data, sr)
                y_aug = apply_time_stretch(y_aug)
                y_aug = apply_add_noise(y_aug)
                new_filename = f"aug_{y}_{i}_{j}.wav"
                new_file_path = os.path.join(output_dir, new_filename)
                sf.write(new_file_path, y_aug, sr)
                new_rows.append({'y': y, 'x': new_file_path})
augmented_rows_df = pd.DataFrame(new_rows)
print(augmented_rows_df.head(10))
print(augmented_rows_df.info())

/home/habib/.venv/lib/python3.14/site-packages/librosa/effects.py:448: FutureWarning: The `hop_length` parameter is deprecated as of 1.0 and will be removed in 1.1. It is unused in the current implementation.
  stft_stretch = core.phase_vocoder(
/home/habib/.venv/lib/python3.14/site-packages/librosa/effects.py:448: FutureWarning: The `n_fft` parameter is deprecated as of 1.0 and will be removed in 1.1. It is unused in the current implementation.
  stft_stretch = core.phase_vocoder(
/home/habib/.venv/lib/python3.14/site-packages/numba/np/ufunc/dufunc.py:303: RuntimeWarning: invalid value encountered in cast
  return super().__call__(*args, **kws)


       y                                                  x
0  tired  /home/habib/mindcloud/project/dataset/augmente...
1  tired  /home/habib/mindcloud/project/dataset/augmente...
2  tired  /home/habib/mindcloud/project/dataset/augmente...
3  tired  /home/habib/mindcloud/project/dataset/augmente...
4  tired  /home/habib/mindcloud/project/dataset/augmente...
5  tired  /home/habib/mindcloud/project/dataset/augmente...
6  tired  /home/habib/mindcloud/project/dataset/augmente...
7  tired  /home/habib/mindcloud/project/dataset/augmente...
8  tired  /home/habib/mindcloud/project/dataset/augmente...
9  tired  /home/habib/mindcloud/project/dataset/augmente...
<class 'pandas.DataFrame'>
RangeIndex: 574 entries, 0 to 573
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   y       574 non-null    str  
 1   x       574 non-null    str  
dtypes: str(2)
memory usage: 9.1 KB
None


In [13]:
# 1. Join both dataframes together vertically
combined_df = pd.concat([train_df, augmented_rows_df], ignore_index=True)

# 2. Randomize the row order completely and reset the index numbers
balanced_train_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

# 3. Check your final balanced class counts
print(balanced_train_df['y'].value_counts())

y
discomfort    308
tired         306
hungry        306
Name: count, dtype: int64


In [14]:
balanced_train_df.head(100)

,y,x
0,tired,/home/habib/mindcloud/project/dataset/tired/C5...
1,tired,/home/habib/mindcloud/project/dataset/augmente...
2,tired,/home/habib/mindcloud/project/dataset/augmente...
3,hungry,/home/habib/mindcloud/project/dataset/hungry/b...
4,tired,/home/habib/mindcloud/project/dataset/augmente...
...,...,...
95,discomfort,/home/habib/mindcloud/project/dataset/augmente...
96,hungry,/home/habib/mindcloud/project/dataset/hungry/D...
97,hungry,/home/habib/mindcloud/project/dataset/hungry/7...
98,tired,/home/habib/mindcloud/project/dataset/augmente...


In [15]:
csv_output_path = "/home/habib/mindcloud/project/balanced_train.csv"
balanced_train_df.to_csv(csv_output_path, index=False)
csv_output_path = "/home/habib/mindcloud/project/test.csv"
test_df.to_csv(csv_output_path, index=False)
csv_output_path = "/home/habib/mindcloud/project/train.csv"
train_df.to_csv(csv_output_path, index=False)
csv_output_path = "/home/habib/mindcloud/project/dataset.csv"
datasetcopy.to_csv(csv_output_path, index=False)
print(f"Successfully saved balanced dataset tracking file to: {csv_output_path}")

Successfully saved balanced dataset tracking file to: /home/habib/mindcloud/project/dataset.csv
